# Sound Processing Lab - Part #4

In this notebook you'll learn:
- How to implement an end-to-end STT-TTS solution

In [ ]:
# Imports and configuration
import os
from dotenv import load_dotenv, find_dotenv
import sounddevice as sd
import scipy.io.wavfile as wav
from transformers import pipeline
import edge_tts
from IPython.display import Audio, display
import google.generativeai as genai
import warnings

print("✅ All audio and AI dependencies successfully loaded!")

## 🛠️ Step 1: Environment Setup & Prerequisites
**Note:** The first run will take time, as it has to download heavy models from the WAN. Be patient.

**How is the flow working:**
Your voice is recorded $\rightarrow$ saved to disk $\rightarrow$ uploaded to Google $\rightarrow$ transcribed $\rightarrow$ sent to the LLM $\rightarrow$ text is sent to Microsoft $\rightarrow$ audio is generated $\rightarrow$ streamed back.

In [ ]:
load_dotenv(find_dotenv())   # load GOOGLE_API_KEY from .env in the project root
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Toggle this to True to run 100% offline on local laptop CPUs
USE_LOCAL_CPU = False

# Configure the official native Google GenAI framework
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

# Using gemini-1.5-flash because it natively accepts direct audio uploads
gemini_model_name = "gemini-2.5-flash"
print("✅ Architecture initialized using native Google Gemini Engine.")

## 🔊 Step 2 — The Ear (Speech-to-Text)
This component uses sounddevice to record physical sound waves straight from the laptop microphone, saves it as a temporary file, and utilizes the blazing-fast Whisper API to transcribe it into clean text tokens.

In [ ]:
def record_microphone(duration=4, sample_rate=16000):
    """Record from the default microphone and save to a WAV file. Returns the saved path."""
    print(f"🎤 Recording for {duration} seconds — speak now!")
    audio_data = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='int16')
    sd.wait()
    print("🛑 Done recording.")
    audio_path = r"..\\stt-tts\\_user_input.wav"
    wav.write(audio_path, sample_rate, audio_data)
    return audio_path


def speech_to_text(audio_file_path, use_local=USE_LOCAL_CPU):
    # Even if the LLM Brain is local, we still use the cloud Ear to process the microphone audio file
    print("👂 ARCHITECTURE STATE: Uploading sound wave to free Google Files API sandbox...")
    
    # Uploads the actual 4-second audio clip recorded from your mic
    audio_payload = genai.upload_file(path=audio_file_path)
    
    model = genai.GenerativeModel(model_name=gemini_model_name)
    
    print("🤖 AI Transcribing your voice...")
    response = model.generate_content([
        "Transcribe this spoken audio exactly. Do not add any greetings, commentary, or introduction. Output ONLY the words spoken.", 
        audio_payload
    ])
    return response.text

## 🧠 Step 3: The Brain (Local SmolLM vs. Cloud Gemini)
This cell evaluates your USE_LOCAL_CPU toggle. It either loads the ultra-lean SmolLM2-360M model straight into your laptop's Python memory sandbox or hands the text prompt over to Google Gemini.

In [ ]:
# Silences deprecation warnings from printing in the notebook
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

def generate_llm_response(prompt, use_local=USE_LOCAL_CPU):
    if use_local:
        print("💻 ARCHITECTURE STATE: Processing reasoning matrix on local CPU via SmolLM2...")
        
        local_pipe = pipeline(
            "text-generation", 
            model="HuggingFaceTB/SmolLM2-360M-Instruct", 
            device=-1
        )
        
        messages = [
            {"role": "system", "content": "You are a witty, ultra-concise voice assistant. Respond in exactly one short sentence."},
            {"role": "user", "content": prompt}
        ]
        
        # Changed clean_up_tokenization_spaces to False to suppress BPE warning
        output = local_pipe(messages, max_new_tokens=50, clean_up_tokenization_spaces=False)
        return output[0]['generated_text'][-1]['content']
        
    else:
        print(f"☁️ ARCHITECTURE STATE: Routing text payload to Cloud Gemini ({gemini_model_name})...")
        model = genai.GenerativeModel(
            model_name=gemini_model_name,
            system_instruction="You are a witty, ultra-concise voice assistant. Respond in exactly one short sentence."
        )
        response = model.generate_content(prompt)
        return response.text

## 🗣️ Step 4: The Voice (Neural Text-to-Speech via edge-tts)
This cell converts the text string answer back into soundwaves. It leverages edge-tts to dodge dependency collisions entirely and renders a native, clickable media player directly inside Jupyter.

In [ ]:
TTS_FILENAME = "..\\stt-tts\\_assistant_output.mp3"

async def text_to_speech(text_input, filename=TTS_FILENAME):
    print("🎨 Synthesizing text sequence into neural audio frequencies...")
    
    # Premium, high-fidelity Microsoft Neural Voice
    active_voice = "en-US-AndrewNeural" 
    
    communicate = edge_tts.Communicate(text_input, active_voice)
    await communicate.save(filename)
    
    # Render native interactive web player tool inside the notebook cell
    display(Audio(filename, autoplay=True))

## 🏎️ Step 5: The Grand Assembly Loop
The final execution architecture cell. Notice the inclusion of the await keyword on step 4; Jupyter executes top-level asynchronous tasks out-of-the-box perfectly.

In [ ]:
# (Ensure your 'record_microphone' function cell has been run before executing this)
try:
    # 1. Capture physical sound wave from microphone
    audio_path = record_microphone(duration=4)
    
    # 2. Step 1: The Ear (Audio -> Text)
    user_text = speech_to_text(audio_path, use_local=USE_LOCAL_CPU)
    print(f"🗣️ Human Said: \"{user_text}\"")
    
    # 3. Step 2: The Brain (Text -> Text Inference)
    ai_text = generate_llm_response(user_text, use_local=USE_LOCAL_CPU)
    print(f"🤖 AI Response: \"{ai_text}\"")
    
    # 4. Step 3: The Voice (Text -> Audio Stream Out)
    await text_to_speech(ai_text)

except Exception as e:
    print(f"\n❌ Pipeline Interrupted: {e}")

# Isn't this cool?  
## What's next you might be thinking...  
### Let's try running the solution entirely local now.


**Steps:**
1. Go to Step 1 and flip the boolean:
```cmd
        USE_LOCAL_CPU = False
```
And make it a True:
```cmd
        USE_LOCAL_CPU = True
```
2. Re-run the Step 1 cell so it takes the change.

3. Now re-run the Step 5 cell and test it.  

<br>  

## Congratulations, you're now running 100% local!  


## Super cool, huh? 🥳 🎉
<br><br><br>


## What have we learned?

Let's discuss in class.  
What's the biggest elephant in the room?  

Yessss.... Latency!!!  

You need to shift from a Batch Sequential Architecture to a Streaming or Native Omnimodal Architecture.  

**1. The Streaming Cascaded Approach (Low Latency)**  
In this setup, you keep the separate Ear, Brain, and Voice components, but instead of waiting for each stage to completely finish, you connect them using WebSockets or WebRTC to stream data in tiny, millisecond-long chunks.

* **Streaming STT (The Ear):** The moment you open your mouth, your microphone sends 100ms chunks of audio to the cloud (using tools like Deepgram or Whisper Streaming). The engine starts transcribing words while you are still mid-sentence.

* **Streaming LLM (The Brain):** As soon as the transcription engine recognizes the first few words, it passes them to the LLM. The LLM doesn't wait for you to finish talking; it begins generating token responses on the fly.

* **Streaming TTS (The Voice):** The moment the LLM spits out its first two or three words, those text tokens are instantly routed to a streaming voice engine (like Cartesia, ElevenLabs, or an advanced edge-tts setup). The speakers begin playing audio before the LLM has even thought of the end of the sentence.  

**The result:** Because these tasks run concurrently, the perceived latency drops from 4–6 seconds down to roughly 500 milliseconds.

<br>  

**2. The Native Audio-to-Audio Approach (Zero Text Bottleneck)**  
This is what models like Gemini Live, Moshi, and Ultravox do. They bypass text conversion entirely.  

Instead of translating sound into text tokens, the neural network is trained natively on Audio Tokens. The model directly maps the raw frequencies of your voice into its latent space and predicts the raw audio frequencies of its response.

**💡 Why this changes everything?**  
Because there is no intermediate step translating back and forth between speech and text, the model doesn't just eliminate latency—it natively understands your tone, laughter, and interruptions, and can respond with its own emotional nuances.

<br>  <br>  <br>  

## Bonus track of the chapter - Let's run a real-time cloud chat session
### Got more time and want to continue playing? Cool!
* Let's do an implementation using Gemini Multimodal Live API (cloud)
* Open Python file: ..\scripts\live_voice_assistant.py
* Follow instructions inside at the top
* Why a separated python file?
    * Standalone script execution prevents background thread collisions with Jupyter's own async loop when interacting with real-time hardware IO. :)


<br>  <br>

Still want some more?
* https://www.youtube.com/watch?v=cvtFdzUQ2Es